In [ ]:
# Group-30 CodeGen — Application UI (voice + auto task routing)
# Two Qwen2.5-Coder-7B LoRA adapters (NL->Python, Python->C++) on one shared base.
# Use a GPU runtime (Runtime -> Change runtime type -> T4/L4 GPU).
!pip -q install gradio "peft>=0.11" "transformers>=4.44" accelerate bitsandbytes

In [ ]:
import os, glob, zipfile, subprocess, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
REPO   = "nayanjha16/CodeGen-Implementations-May_26"
BRANCH = "Group-30_dev"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ADAPTERS = {
    "NL -> Python":  "Qwen_NL_to_PL_LORA_Adapter",
    "Python -> C++": "Qwen_Python_to_CPP_LORA_Adapter",
}
WORK = "/content/adapters"; os.makedirs(WORK, exist_ok=True)

In [ ]:
# Fetch each adapter's REAL zip from the LFS media URL into WORK, then unzip.
def ensure_adapter(folder):
    out_dir  = os.path.join(WORK, folder)
    if os.path.exists(os.path.join(out_dir, "adapter_config.json")):
        print("Already unzipped:", out_dir); return out_dir
    zip_path = os.path.join(WORK, folder + ".zip")
    url = f"https://media.githubusercontent.com/media/{REPO}/{BRANCH}/Group-30/src/model/{folder}.zip"
    print("Downloading", folder, "...")
    r = subprocess.run(["curl", "-fL", url, "-o", zip_path], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"download failed (curl {r.returncode}): {r.stderr[:300]}")
    size = os.path.getsize(zip_path)
    if size < 100_000:
        raise RuntimeError(f"{folder}.zip is only {size} bytes — not the real adapter.")
    print("Unzipping %s (%.0f MB)" % (folder, size/1e6))
    with zipfile.ZipFile(zip_path) as z: z.extractall(WORK)
    return out_dir

ADAPTER_PATHS = {task: ensure_adapter(folder) for task, folder in ADAPTERS.items()}
print(ADAPTER_PATHS)

In [ ]:
# Load base ONCE (4-bit), attach BOTH adapters. Guarded so a re-run won't reload the 7B.
TASKS = list(ADAPTERS.keys())
if "model" not in globals():
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tok.pad_token_id is None: tok.pad_token_id = tok.eos_token_id
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto",
                                                quantization_config=bnb, torch_dtype=torch.bfloat16)
    first, *rest = TASKS
    model = PeftModel.from_pretrained(base, ADAPTER_PATHS[first], adapter_name=first)
    for t in rest: model.load_adapter(ADAPTER_PATHS[t], adapter_name=t)
    model.eval()
    print("Loaded base + adapters:", TASKS)
else:
    print("Model already loaded — reusing.")

In [ ]:
# Speech-to-text (downloads whisper-base into Colab).
asr = pipeline("automatic-speech-recognition", model="openai/whisper-base",
                device=0 if torch.cuda.is_available() else -1)
print("Whisper loaded.")

In [ ]:
def build_prompt(task, text):
    if task == "Python -> C++":
        user = ("Convert the following Python code to C++. Return only the C++ code, "
                "no explanation.\n\n```python\n" + text.strip() + "\n```")
    else:
        user = ("Generate Python code for the following description. Return only the "
                "Python code, no explanation.\n\n" + text.strip())
    messages = [{"role": "system", "content": "You are Qwen, a helpful coding assistant."},
                {"role": "user", "content": user}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def extract_code(text):
    text = text.strip()
    if "```" in text:
        parts = text.split("```")
        if len(parts) >= 2:
            block = parts[1]; lines = block.split("\n")
            if lines and lines[0].strip().lower() in ("python", "py", "cpp", "c++"):
                block = "\n".join(lines[1:])
            return block.strip()
    return text

@torch.no_grad()
def generate(task, text):
    if not text or not text.strip(): return "// enter some input first"
    model.set_adapter(task)
    inputs = tok(build_prompt(task, text), return_tensors="pt",
                  truncation=True, max_length=2048).to(DEVICE)
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False,
                          pad_token_id=tok.eos_token_id)
    new = out[0][inputs["input_ids"].shape[1]:]
    return extract_code(tok.decode(new, skip_special_tokens=True))

@torch.no_grad()
def classify_task(text):
    if not text or not text.strip(): return "NL -> Python"
    user = ("You are a router. Reply with EXACTLY one label, nothing else:\n"
            "  NL2PY  = input is a natural-language description of what code should do\n"
            "  PY2CPP = input is existing Python source code to convert to C++\n\n"
            f"Input:\n{text.strip()}\n\nLabel:")
    messages = [{"role": "system", "content": "You are Qwen, a helpful assistant."},
                {"role": "user", "content": user}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(DEVICE)
    with model.disable_adapter():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False,
                              pad_token_id=tok.eos_token_id)
    ans = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).upper()
    return "Python -> C++" if "PY2CPP" in ans else "NL -> Python"

def transcribe(audio_path):
    if not audio_path: return ""
    return asr(audio_path, chunk_length_s=30)["text"].strip()

In [ ]:
import gradio as gr
AUTO = "Auto (detect)"
UI_TASKS = [AUTO] + TASKS

def run(task_choice, audio_path, text):
    if audio_path:
        spoken = transcribe(audio_path)
        if spoken: text = spoken
    if not text or not text.strip():
        return "", "enter text or record audio first", gr.update()
    resolved = classify_task(text) if task_choice == AUTO else task_choice
    code = generate(resolved, text)
    lang = "cpp" if resolved == "Python -> C++" else "python"
    return text, f"Task used: {resolved}", gr.update(value=code, language=lang)

with gr.Blocks(title="Group-30 CodeGen") as demo:
    gr.Markdown("# Group-30 CodeGen\nType **or speak**. Leave task on Auto and it picks the model for you.")
    task = gr.Dropdown(UI_TASKS, value=AUTO, label="Task")
    with gr.Row():
        mic = gr.Audio(sources=["microphone"], type="filepath", label="Speak (optional)")
    inp = gr.Textbox(lines=8, label="Input (or transcribed speech)",
                      placeholder="Describe the function, or paste Python code...")
    btn = gr.Button("Generate", variant="primary")
    status = gr.Markdown()
    out = gr.Code(label="Output", language="python")
    btn.click(run, [task, mic, inp], [inp, status, out])

demo.launch(share=True)